# Count the observations and preserve their identities

This notebook audits the complete annotation inventory, retained source groups, observation times, and external-data eligibility before interpreting any performance.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Rebuild the annotation inventory

One source video can contribute several sequences, and each sequence can contribute overlapping windows. We count these units separately because windows from one recording do not establish independent participants. The transformed sequence files are the annotation inventory; the upstream source CSVs are a provenance copy and are excluded from this count.

In [ ]:
from worldmodels.v2_data import all_annotations
catalog = all_annotations(REPOSITORY)
USED_INPUTS.update(dict(zip(catalog.csv_path,catalog.csv_sha256)))
display(catalog.groupby('label').agg(sequences=('sequence_id','nunique'), rows=('annotation_rows','sum'), sources=('source_id','nunique')))
assert len(catalog)==1874 and catalog.annotation_rows.sum()==458116 and catalog.source_id.nunique()==348
print('Annotated normal source ceiling:',catalog.loc[catalog.label.eq('normal'),'source_id'].nunique())


## Follow the population through the gates

The study reuses all eligible observed spans from the five selected annotations. The unchanged observation and target-support gates define a common population for every model. A gate can select a different fraction of each annotation, so the table includes both retained coverage and attrition.

In [ ]:
audit = read_json(COHORT/'audit.json')
display(read_csv(COHORT/'attrition_by_label.csv'))
display(metadata.groupby('label').agg(windows=('window_id','size'),sequences=('sequence_id','nunique'),sources=('source_id','nunique')))
assert audit['prepared_windows']==2818 and audit['model_windows']==2815
assert audit['reconstruction']['maximum_absolute_error']==0
print('Full-span windows were reconstructed from sequence poses without changing the frozen arrays.')


## Keep related observations together

The relation graph uses source identity and verified byte-identical footage. A copy under another local dataset folder remains the same recording. Different file hashes do not prove that the people are different. Unknown repeated participants therefore remain a generalization limit.

In [ ]:
relations = read_csv(COHORT/'media_relations.csv')
copies = relations.groupby('source_id').agg(local_files=('path','size'),different_hashes=('sha256','nunique'))
display(copies.query('local_files > 1').head(12))
graph = read_json(COHORT/'relation_graph.json')
print('Verified cross-source footage edges:',graph['verified_cross_source_edges'])
mixed=catalog.groupby('source_id').label.agg(lambda values:sorted(set(values)))
display(mixed[mixed.map(len)>1].rename('annotation_labels').to_frame())
print('Current model sources/components:',metadata.source_id.nunique(),metadata.component_id.nunique())
assert metadata.groupby('component_id').outer_fold.nunique().eq(1).all()
display(metadata[['source_id','component_id','label','outer_fold']].drop_duplicates().groupby(['outer_fold','label']).size().unstack(fill_value=0))


## Check the time axis and observation support

The nominal rate is ten samples per second, but every sample is an actual decoded frame. The final forecast extends about 0.8 seconds beyond the last observed frame. Future observation support is used after prediction to score available outcomes, while prefix measurements determine the predictor's inputs.

In [ ]:
gaps = np.diff(actual_times,axis=1)
display(metadata[['sample_fps','prefix_valid_fraction','future_valid_fraction','final_horizon_seconds','maximum_grid_error_seconds']].describe())
print('Actual decoder interval range:',float(gaps.min()),float(gaps.max()),'seconds')
assert np.isfinite(actual_times).all() and (gaps>0).all()
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
fig,ax=plt.subplots(figsize=(10,4.5))
ax.imshow(valid[0].T,aspect='auto',interpolation='nearest',cmap='Greens',vmin=0,vmax=1)
ax.axvline(15.5,color='#a45d21',lw=2)
ax.set_yticks(range(16),labels=[11,12,13,14,15,16,23,24,25,26,27,28,29,30,31,32])
ax.set(xlabel='Sample index; the future begins at 16',ylabel='BlazePose joint ID',title='Natural support in the first retained window in the pinned membership table')
fig.legend(handles=[Patch(facecolor='#00441b',label='Measured joint above the confidence threshold'),
                    Patch(facecolor='#f7fcf5',edgecolor='#888888',label='Naturally missing joint'),
                    Line2D([0],[0],color='#a45d21',label='Prefix/future boundary')],
           loc='lower center',bbox_to_anchor=(.5,-.21),ncol=1,frameon=False,fontsize=10)
save_figure(fig,'20_observed_support');plt.show()


## Identify the external transfer population

The external dataset is evaluated with frozen source-trained models after a separate compatibility audit. Recording names are mapped to participant codes using their annotation context. The external population is reported separately from GAVD, and its different acquisition conditions remain visible.

In [ ]:
external = read_json(PROJECT/'docs/reference_v2/data_external_summary.json')
display(external)
display(read_csv(PROJECT/'docs/reference_v2/data_external_inventory.csv').head())
print('GAVD is a heterogeneous source-video benchmark; participant and site independence are not established by its source IDs.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/20_cohort_identity_and_coverage.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
